In [1]:
import sympy
x, y = sympy.symbols('x y')

f = sympy.sin(x + y)**2 + 5 + sympy.cos(y)**2 - sympy.exp(-(x**2 + y**2 + 146*y + 54*x + 6058) / 25)

df_dx = sympy.diff(f, x)
df_dy = sympy.diff(f, y)
h = sympy.hessian(f, (x, y))

print(df_dx)
print(df_dy)
print(h)

-(-2*x/25 - 54/25)*exp(-x**2/25 - 54*x/25 - y**2/25 - 146*y/25 - 6058/25) + 2*sin(x + y)*cos(x + y)
-(-2*y/25 - 146/25)*exp(-x**2/25 - 54*x/25 - y**2/25 - 146*y/25 - 6058/25) - 2*sin(y)*cos(y) + 2*sin(x + y)*cos(x + y)
Matrix([[-(-2*x/25 - 54/25)**2*exp(-x**2/25 - 54*x/25 - y**2/25 - 146*y/25 - 6058/25) + 2*exp(-x**2/25 - 54*x/25 - y**2/25 - 146*y/25 - 6058/25)/25 - 2*sin(x + y)**2 + 2*cos(x + y)**2, -(-2*x/25 - 54/25)*(-2*y/25 - 146/25)*exp(-x**2/25 - 54*x/25 - y**2/25 - 146*y/25 - 6058/25) - 2*sin(x + y)**2 + 2*cos(x + y)**2], [-(-2*x/25 - 54/25)*(-2*y/25 - 146/25)*exp(-x**2/25 - 54*x/25 - y**2/25 - 146*y/25 - 6058/25) - 2*sin(x + y)**2 + 2*cos(x + y)**2, -(-2*y/25 - 146/25)**2*exp(-x**2/25 - 54*x/25 - y**2/25 - 146*y/25 - 6058/25) + 2*exp(-x**2/25 - 54*x/25 - y**2/25 - 146*y/25 - 6058/25)/25 + 2*sin(y)**2 - 2*sin(x + y)**2 - 2*cos(y)**2 + 2*cos(x + y)**2]])


In [2]:
import numpy as np
import matplotlib.pyplot as plt

def f(x, y):
    
    return np.sin(x + y)**2 + 5 + np.cos(y)**2 - np.exp(-(x**2 + y**2 + 146*y + 54*x + 6058) / 25)

def gradient(x, y):

    df_dx = -(-2*x/25 - 54/25)*np.exp(-x**2/25 - 54*x/25 - y**2/25 - 146*y/25 - 6058/25) + 2*np.sin(x + y)*np.cos(x + y)
    df_dy = -(-2*y/25 - 146/25)*np.exp(-x**2/25 - 54*x/25 - y**2/25 - 146*y/25 - 6058/25) - 2*np.sin(y)*np.cos(y) + 2*np.sin(x + y)*np.cos(x + y)
    return np.array([df_dx, df_dy])

def backtracking_search(x, y, dir_x, dir_y):
    grad = gradient(x, y)
    alpha, beta = 0.5, 0.5
    t = 1
    while f(x + dir_x*t, y + dir_y*t) > f(x, y) + alpha * t * (grad[0]*dir_x + grad[1]*dir_y):
        t *= beta
    return t 

def gradient_method_algorithm(x, y, max_iterations=1000, epsilon=1e-6, record_path=False):
    
    path = [(x, y)] if record_path else None
    
    for i in range(max_iterations):
        grad = gradient(x, y)
        if np.linalg.norm(grad) < epsilon:
            break
        dir_x, dir_y = -grad[0], -grad[1]
        t = backtracking_search(x, y, dir_x, dir_y)
        x += t * dir_x
        y += t * dir_y
        
        if record_path:
            path.append((x, y))
    
    if record_path:
        return x, y, f(x, y), path
    else:
        return x, y, f(x, y)

def find_global_minimum(num_restarts=500, domain=[-100, 100]):

    all_results = []
    best_point = None
    best_value = np.inf
    
    for i in range(num_restarts):
        start_x = np.random.uniform(domain[0], domain[1])
        start_y = np.random.uniform(domain[0], domain[1])
        
        x_opt, y_opt, value = gradient_method_algorithm(start_x, start_y)
        all_results.append({
            'start': (start_x, start_y),
            'optimum': (x_opt, y_opt),
            'value': value
        })
        
        if value < best_value:
            best_value = value
            best_point = (x_opt, y_opt)
    
    return best_point, best_value, all_results


def plot_trajectories(best_point, all_results, domain=[-5000, 5000], num_trajs=30):
    
    x_vals = np.linspace(domain[0], domain[1], 400)
    y_vals = np.linspace(domain[0], domain[1], 400)
    X, Y = np.meshgrid(x_vals, y_vals)
    Z = f(X, Y)

    plt.figure(figsize=(10, 8))
    plt.contourf(X, Y, Z, levels=80, cmap='viridis')
    plt.colorbar(label='f(x, y)')
    
    subset = np.random.choice(len(all_results), size=min(num_trajs, len(all_results)), replace=False)
    for i in subset:
        start_x, start_y = all_results[i]['start']
        _, _, _, path = gradient_method_algorithm(start_x, start_y, record_path=True)
        path = np.array(path)
        plt.plot(path[:,0], path[:,1], color='white', alpha=0.6, linewidth=1)
        plt.scatter(path[0,0], path[0,1], color='red', s=10)  
        plt.scatter(path[-1,0], path[-1,1], color='cyan', s=10)  

    plt.scatter(best_point[0], best_point[1], color='gold', s=100, marker='*', label='Global minimum')
    plt.title("Trajectories of gradient descent")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.legend()
    plt.show()


best_point, best_value, all_results = find_global_minimum(num_restarts=200)
print("Global minimum", best_point, "value of function:", best_value)
print(all_results)

#plot_trajectories(best_point, all_results, domain=[-5000, 5000])


Global minimum (np.float64(-32.871284032742906), np.float64(-70.75286569609992)) value of function: 4.801030612347589
[{'start': (-57.330134351817264, 67.48424196106781), 'optimum': (np.float64(-58.11946302073213), np.float64(67.54424146143968)), 'value': np.float64(5.000000000000579)}, {'start': (56.79106585296046, -56.82798399895572), 'optimum': (np.float64(58.11946321095975), np.float64(-58.11946362892867)), 'value': np.float64(5.000000000000389)}, {'start': (-98.5512668960864, -67.81837305235432), 'optimum': (np.float64(-98.96016798687015), np.float64(-67.5442424790876)), 'value': np.float64(5.000000000000212)}, {'start': (10.586536845708828, -46.35871413630805), 'optimum': (np.float64(10.995573540418057), np.float64(-45.55309296655947)), 'value': np.float64(5.000000000000316)}, {'start': (-62.591105334218945, 82.39455654526064), 'optimum': (np.float64(-64.40264886898618), np.float64(83.25220492977047)), 'value': np.float64(5.000000000000172)}, {'start': (24.92291463008971, 71.3134

In [3]:
global_min_point, global_min_value, starting_points = find_global_minimum(
    num_restarts=5000, 
    domain = [-5000, 5000]
)
print(f"Global minimum at: {global_min_point} with value {global_min_value}")

Global minimum at: (np.float64(-3020.6413359236776), np.float64(1635.1989759418489)) with value 5.000000000000126
